## Library 불러오기

In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
from tqdm import tqdm
import time

## 보조 함수: Correalation 계산 관련

### Sparse Matrix에서 특정 인덱스 행 제거하는 함수[1]

In [ ]:
def delete_rows_csr(mat, idx, *additial_idx):
    if not isinstance(mat, sp.sparse.csr_matrix):
        raise ValueError("works only for CSR format -- use .tocsr() first")
    mask = np.ones(mat.shape[0], dtype=bool)
    indices = list(idx)
    mask[indices] = False
    for _idx in additial_idx:
        indices = list(_idx)
        mask[indices] = False
    return mat[mask,:]

### Sparse Matrix에서 각 행에서 0이 아닌 값들에 대한 분산과 평균을 구하는 함수[2]

In [ ]:
def var_sparse_mat(A, axis=None, nonzero=True):
    A_nonzero = (A != 0).sum(axis)
    A_squared = A.copy()
    A_squared.data **= 2
    A_squared_mean = np.divide(A_squared.sum(axis), A_nonzero) if nonzero else A_squared.mean(axis)
    A_mean_square = np.square(np.divide(A.sum(axis), A_nonzero)) if nonzero else np.square(A.mean(axis))
    return A_squared_mean - A_mean_square

In [ ]:
def mean_sparse_mat(A, axis=None, nonzero=True):
    A_nonzero = (A != 0).sum(axis)
    return np.divide(A.sum(axis), A_nonzero) if nonzero else A.mean(axis)

### 두 Sparse Matrix의 각 행 벡터의 Correlation Coefficient 계산하는 함수[3]

In [ ]:
def corrcoef_sparse_mat_vec(A, B, len_vec):
    # Store row-wise in A and B, as they would be used at few places
    sA = np.asarray(A.sum(axis=1))
    sB = np.asarray(B.sum(axis=1))
    
    # Basically there are four parts in the formula. We would compute them one-by-one
    p1 = np.multiply(np.asarray(A.multiply(B).sum(axis=1)), len_vec.reshape(-1, 1))
    p2 = np.multiply(sA, sB)
    p3 = np.multiply(B.power(2).sum(axis=1), len_vec.reshape(-1, 1)) - (sB ** 2)
    p4 = np.multiply(A.power(2).sum(axis=1), len_vec.reshape(-1, 1)) - (sA ** 2)
    
    # Finally compute Pearson Correlation Coefficient as 1D array
    pcorr = np.asarray(np.divide(p1 - p2, np.sqrt(np.multiply(p3, p4)))).reshape(-1)

    return pcorr

## 보조함수: Recommendation 관련

### 내림차순 정렬된 Dictionary에서 누적 비율이 특정 수치보다 높아지는 Key의 순위를 반환하는 함수 

In [ ]:
def find_opt_key_num(sorted_dict, cut):
    value_sum = 0
    ret = 0
    for value in list(sorted_dict.values()):
        value_sum = value_sum + value
        ret = ret + 1
        if value_sum >= cut:
            return ret

### 데이터 내 영어 장르이름을 한국어로 변환하는 함수 

In [ ]:
def get_kor_genre_name(genre_nm):
    if genre_nm == 'adventure':
        return '어드밴처'
    elif genre_nm == 'action':
        return '액션'
    elif genre_nm == 'family':
        return '가족'
    elif genre_nm == 'romance':
        return '로맨스'
    elif genre_nm == 'comedy':
        return '코미디'
    elif genre_nm == 'drama':
        return '드라마'
    elif genre_nm == 'thriller':
        return '스릴러'
    elif genre_nm == 'mystery':
        return '미스터리'
    elif genre_nm == 'horror':
        return '호러'
    elif genre_nm == 'war':
        return '전쟁'
    elif genre_nm == 'wuxia':
        return '무협'
    elif genre_nm == 'musical':
        return '뮤지컬'
    elif genre_nm == 'historical':
        return '시대극'
    elif genre_nm == 'sports':
        return '스포츠'
    elif genre_nm == 'fantasy':
        return '판타지'
    elif genre_nm == 'sf':
        return 'SF'
    elif genre_nm == 'cooking':
        return '요리'
    elif genre_nm == 'hero':
        return '히어로'
    elif genre_nm == 'idol':
        return '아이돌'
    elif genre_nm == 'school':
        return '학원물'
    else:
        return '기타'

### CF 알고리즘 추천이 가능한 유저 (조건부) 임의 추출 함수
<code>conds</code>는 하나 이상의 조건을 포함하는 문자열이며, 각 조건은 콤마(, Comma)로 분류해야합니다.  
하나의 조건은 \<열 이름\> \<(부)등호\> \<수치\>의 양식을 가지고 있어야합니다.  
예를 들어, 시청한 애니메이션이 겹치는 개수가 평균 5회 이상이고, Correlation 계산에 이용된 유저 수가 200명 이상이라는 조건은 아래와 작성하면 됩니다.  
'AVG_OVERLAP >= 5, NUM_CORR_USER >= 200'

In [ ]:
def draw_random_user(cf_fs_df, n=1, conds=''):
    if conds == '':
        return cf_fs_df[cf_fs_df['FAIL'] == 0]['svc_mgmt_num'].sample(n=n).tolist()
    else:
        final_cond = (cf_fs_df['FAIL'] == 0)
        cond_list = conds.split(',')
        for cond in cond_list:
            cond = cond.strip()
            column_name = cond.split(' ')[0]
            eq = cond.split(' ')[1]
            num = cond.split(' ')[2]
            if column_name not in [col for col in cf_fs_df.columns if col != 'svc_mgmt_num' and col != 'FAIL' and col != 'FAIL_REASON']:
                print(column_name + ': 유효하지 않은 칼럼 이름입니다. 아래의 칼럼 중 하나를 입력해주세요.')
                print(', '.join([col for col in cf_fs_df.columns if col != 'svc_mgmt_num' and col != 'FAIL' and col != 'FAIL_REASON']))
                return []
    
            if eq != '==' and eq != '>=' and eq != '<=' and eq != '>' and eq != '<':
                print(eq + ': 지원하지 않는 비교 연산자입니다. 아래의 비교 연산자 중 하나를 입력해주세요.')
                print('==, >=, <=, >, <')
                return []
    
            if num.isdigit():
                num_ = int(num)
            else:
                num_ = float(num)

            if eq == '==':
                final_cond = final_cond & (cf_fs_df[column_name] == num_)
            elif eq == '>=':
                final_cond = final_cond & (cf_fs_df[column_name] >= num_)
            elif eq == '<=':
                final_cond = final_cond & (cf_fs_df[column_name] <= num_)
            elif eq == '<':
                final_cond = final_cond & (cf_fs_df[column_name] < num_)
            else:
                final_cond = final_cond & (cf_fs_df[column_name] > num_)

        return cf_fs_df[final_cond]['svc_mgmt_num'].sample(n=n).tolist()

## 데이터 불러오기

In [ ]:
ani_rating_df = pd.read_csv('./ani_rating.csv') # 평점 행렬 (유저 by 애니메이션)

## 데이터프레임을 Sparse Matrix로 변환

In [ ]:
raw_mat = ani_rating_df[[col for col in ani_rating_df.columns if col != 'svc_mgmt_num']].copy(True).to_numpy()
CF_mat_one = sp.sparse.csr_matrix(np.where((raw_mat != 1) & (raw_mat != 0), 1, raw_mat))
CF_mat = sp.sparse.csr_matrix(raw_mat)
CS_mat = ani_rating_df['svc_mgmt_num'].to_numpy()

### 유저 수와 애니메이션 수 확인

In [ ]:
user_num = np.shape(CF_mat)[0]
ani_num = np.shape(CF_mat)[1]
print('user_num: {0}'.format(user_num))
print('ani_num: {0}'.format(ani_num))

### 각 애니메이션에 대해, Rating을 남긴 유저 인덱스 값 저장

In [ ]:
positive_rating_idx_dict = {} # 'ani_col_idx: list of index'
for col in range(CF_mat.shape[1]):
    positive_rating_idx_dict[col] = sp.sparse.find(CF_mat[:, col] > 0)

## CF Feasibility 확인 및 Correaltion 계산 함수
### Input
<code>i</code>: Sparse Matrix에서 행을 나타내는 Index 값입니다.  
<code>min_overlap</code>: 시청한 애니메이션 중 겹치는 개수의 최솟값입니다. 기본값은 3입니다.  
<code>std</code>: 평점을 표준화하는데 있어 필요한 정보를 추가로 저장할지를 결정합니다. 기본값은 False입니다.  
<code>export_mode</code>: Feasibility 데이터 추출에 필요없는 정보들은 결과값에 저장하지 않습니다. 기본값은 False입니다.  
### Output
<code>ret</code>: Dictionary 형태의 데이터입니다. 'SUMMARY_STAT'키에는 겹친 애니메이션 개수와 계산된 Correlation에 대한 요약통계량을 담고 있으며, 'RATING_STAT'에는 Rationg과 관련된 요약 통계량이, 'OVERLAP_DATA'키에는 다른 유저와의 Correlation 데이터가 저장됩니다.

In [ ]:
def get_cf_feasibility(i, min_overlap=3, std=False, export_mode=False):
    ret = {}

    this_CF_mat = CF_mat[i]
    this_CF_mat_one = CF_mat_one[i]
    this_user_id = CS_mat[i]
    
    fail = False
    # 해당 유저의 Implicit Rating의 분산이 0일 경우 계산 불가
    this_user_var = var_sparse_mat(this_CF_mat, axis=1)
    if this_user_var == 0:
        fail = True
        num_common_user = np.nan
        avg_common_ani_num = np.nan
        std_common_ani_num = np.nan
        pair_string = ""
        cf_fail_reason = "No variance in implicit rating"
    else:
        this_user_mat = CF_mat.multiply(this_CF_mat_one)
        this_user_mat_one = CF_mat_one.multiply(this_CF_mat_one)
        this_user_vec = np.asarray(this_user_mat_one.sum(axis=1))
        if std or export_mode:
            this_rating_mean = np.asarray(mean_sparse_mat(this_user_mat[i], axis=1)).reshape(-1).item()
            this_rating_std = np.asarray(np.sqrt(var_sparse_mat(this_user_mat[i], axis=1))).reshape(-1).item()

        # 겹치지 않는 유저들은 삭제
        no_overlap_user_idx = np.append(np.where(this_user_vec == 0)[0], i)
        this_user_mat = delete_rows_csr(this_user_mat, no_overlap_user_idx)
        this_user_mat_one = delete_rows_csr(this_user_mat_one, no_overlap_user_idx)
        this_user_vec = np.delete(this_user_vec, no_overlap_user_idx)
        this_user_num = this_user_mat.shape[0]
        this_other_user_id_mat = np.delete(CS_mat, no_overlap_user_idx)
        
        # 조건 1. 시청한 애니메이션이 겹치는 다른 유저들 중 최소 한 유저는 평점 분산이 0보다 크다
        var_mat = var_sparse_mat(this_user_mat, axis=1)
        filter_idx = sp.sparse.find((var_mat == 0) | (var_mat == np.nan))[0]
        if len(filter_idx) == this_user_num:
            fail = True
            num_common_user = np.nan
            avg_common_ani_num = np.nan
            std_common_ani_num = np.nan
            pair_string = ""
            cf_fail_reason = "No variance in implict ratings of the others who overlap in watched animations"
        else:
            # 조건 2. 다른 유저들의 겹치는 애니메이션의 평점의 분산이 0보다 크다
            this_user_mat_overlap = this_user_mat_one.multiply(CF_mat[i]) # 대상 유저와 겹치는 애니메이션 평점만 남김
            filter_idx = np.union1d(filter_idx, sp.sparse.find(var_sparse_mat(this_user_mat_overlap, axis=1) == 0)[0])
            if len(filter_idx) == this_user_num:
                fail = True
                num_common_user = np.nan
                avg_common_ani_num = np.nan
                std_common_ani_num = np.nan
                pair_string = ""
                cf_fail_reason = "No variance in implict ratings of the others who overlap in watched animations"
            else:
                # 조건 3. 겹치는 애니메이션의 수가 최소 정한 수치 (min_overlap) 이상이어야 한다
                filter_idx = np.union1d(filter_idx, np.where(this_user_vec < min_overlap)[0])
                if len(filter_idx) == this_user_num:
                    fail = True
                    num_common_user = np.nan
                    avg_common_ani_num = np.nan
                    std_common_ani_num = np.nan
                    pair_string = ""
                    cf_fail_reason = "Not enough the number of others who overlap in watched animations"
                else:
                    ret['SUMMARY_STAT'] = {}
                    # 조건에 맞지 않은 유저들 및 해당 유저는 모두 계산되는 행렬에서 삭제
                    this_user_mat = delete_rows_csr(this_user_mat, filter_idx)
                    this_user_mat_overlap = delete_rows_csr(this_user_mat_overlap, filter_idx)
                    this_user_mat_one = delete_rows_csr(this_user_mat_one, filter_idx)
                    this_user_vec = np.delete(this_user_vec, filter_idx).reshape(-1)
                    this_other_user_id_mat = np.delete(this_other_user_id_mat, filter_idx)
                    
                    corr_array = corrcoef_sparse_mat_vec(this_user_mat, this_user_mat_overlap, this_user_vec)

                    if std or export_mode:
                        ret['RATING_STAT'] = {}
                        ret['RATING_STAT']['THIS_RATING'] = (this_rating_mean, this_rating_std)
                        if not export_mode:
                            o_mean = np.asarray(mean_sparse_mat(this_user_mat, axis=1)).reshape(-1).tolist()
                            o_std = np.asarray(np.sqrt(var_sparse_mat(this_user_mat, axis=1))).reshape(-1).tolist()
                            ret['RATING_STAT']['OTHER_RATING'] = dict(zip(this_other_user_id_mat.tolist(), zip(o_mean, o_std)))

                    if not export_mode:
                        ret['OVERLAP_DATA'] = dict(zip(this_other_user_id_mat.tolist(), corr_array.tolist()))
                    ret['SUMMARY_STAT']['AVG_OVERLAP'] = np.mean(this_user_vec)
                    ret['SUMMARY_STAT']['STD_OVERLAP'] = np.std(this_user_vec)                   
                    ret['SUMMARY_STAT']['MIN_OVERLAP'] = np.min(this_user_vec)
                    ret['SUMMARY_STAT']['MAX_OVERLAP'] = np.max(this_user_vec)
                    ret['SUMMARY_STAT']['NUM_CORR_USER'] = len(this_user_vec)
                    ret['SUMMARY_STAT']['AVG_CORR'] = np.mean(corr_array)
                    ret['SUMMARY_STAT']['STD_CORR'] = np.std(corr_array)

    ret['FAIL'] = 1 if fail else 0
    if fail:
        ret['FAIL_REASON'] = cf_fail_reason
    
    return ret

## Feasibility 데이터를 CSV 파일로 추출

In [ ]:
summary_df = {}
summary_df['RATING_MEAN'] = {}
summary_df['RATING_STD'] = {}
summary_df['AVG_OVERLAP'] = {}
summary_df['MIN_OVERLAP'] = {}
summary_df['MAX_OVERLAP'] = {}
summary_df['NUM_CORR_USER'] = {}
summary_df['AVG_CORR'] = {}
summary_df['STD_CORR'] = {}
summary_df['FAIL'] = {}
summary_df['FAIL_REASON'] = {}

In [ ]:
for i in tqdm(range(user_num)):
    ret = get_cf_feasibility(i, export_mode=True)
    if ret['FAIL']:
        summary_df['RATING_MEAN'].update({CS_mat[i]: np.nan})
        summary_df['AVG_OVERLAP'].update({CS_mat[i]: np.nan})
        summary_df['MIN_OVERLAP'].update({CS_mat[i]: np.nan})
        summary_df['MAX_OVERLAP'].update({CS_mat[i]: np.nan})
        summary_df['NUM_CORR_USER'].update({CS_mat[i]: np.nan})
        summary_df['AVG_CORR'].update({CS_mat[i]: np.nan})
        summary_df['STD_CORR'].update({CS_mat[i]: np.nan})
        summary_df['FAIL'].update({CS_mat[i]: 1})
        summary_df['FAIL_REASON'].update({CS_mat[i]: ret['FAIL_REASON']})
        continue

    summary_df['RATING_MEAN'].update({CS_mat[i]: ret['RATING_STAT']['THIS_RATING'][0]})
    summary_df['RATING_STD'].update({CS_mat[i]: ret['RATING_STAT']['THIS_RATING'][1]})
    summary_df['AVG_OVERLAP'].update({CS_mat[i]: ret['SUMMARY_STAT']['AVG_OVERLAP']})
    summary_df['MIN_OVERLAP'].update({CS_mat[i]: ret['SUMMARY_STAT']['MIN_OVERLAP']})
    summary_df['MAX_OVERLAP'].update({CS_mat[i]: ret['SUMMARY_STAT']['MAX_OVERLAP']})
    summary_df['NUM_CORR_USER'].update({CS_mat[i]: ret['SUMMARY_STAT']['NUM_CORR_USER']})
    summary_df['AVG_CORR'].update({CS_mat[i]: ret['SUMMARY_STAT']['AVG_CORR']})
    summary_df['STD_CORR'].update({CS_mat[i]: ret['SUMMARY_STAT']['STD_CORR']})
    summary_df['FAIL'].update({CS_mat[i]: 0})
    summary_df['FAIL_REASON'].update({CS_mat[i]: ''})

In [ ]:
cf_feasibility_df = pd.DataFrame(data=summary_df)
cf_feasibility_df.index.name = 'svc_mgmt_num'

In [ ]:
cf_feasibility_df.to_csv('./cf_feasibility.csv')

In [ ]:
print('CF로 추천 가능한 유저 수: {0} (비율: {1})'.format(len(cf_feasibility_df[cf_feasibility_df['FAIL'] == 0]),
                                              len(cf_feasibility_df[cf_feasibility_df['FAIL'] == 0]) / len(cf_feasibility_df) ))
print('CF로 추천 가능한 유저 중에서')
print('평균 Implict Rating의 평균: {0}'.format(cf_feasibility_df['RATING_MEAN'].mean()))
print('Implict Rating의 표준편차의 평균: {0}'.format(cf_feasibility_df['RATING_STD'].mean()))
print('CF에 이용된 겹친 시청한 애니메이션 개수의')
print('평균의 평균: {0}'.format(cf_feasibility_df['AVG_OVERLAP'].mean()))
print('최솟값의 평균: {0}'.format(cf_feasibility_df['MIN_OVERLAP'].mean()))
print('최댓값의 평균: {0}'.format(cf_feasibility_df['MAX_OVERLAP'].mean()))
print('평균 Correlation의 평균: {0}'.format(cf_feasibility_df['AVG_CORR'].mean()))
print('Correlation의 표준편차의 평균: {0}'.format(cf_feasibility_df['STD_CORR'].mean()))

## CF로 애니메이션 추천

In [ ]:
#cf_feasibility_df = pd.read_csv('./cf_feasibility_df.csv')
ani_meta_df = pd.read_csv('./contents_list_2156.csv', index_col='sris_id')

In [ ]:
genre_cols = [col for col in ani_meta_df.columns if col.startswith('gnr_') == True]
def gnr_extract(x):
    for col in genre_cols:
        if x[col] == 1:
            return col.split('_')[1]

for idx, ani in ani_meta_df.iterrows():
    ani_meta_df.loc[idx, 'gnr'] = gnr_extract(ani)

all_ani = [(ani_id, ani_rating_df.columns.get_loc(ani_id) - 1) for ani_id in ani_rating_df.columns if ani_id != 'svc_mgmt_num']

In [ ]:
def recommendation(user_id, output_len=1, std=True, min_ir_user_num=5, cut=0.6, min_prop=0.1, verbose=False):
    if cut <= 0 or cut > 1:
        print('잘못된 Parameter 요청. cut은 반드시 0과 1 사이의 값이어야합니다.')
        return 'None'

    if min_prop < 0 or min_prop > 1:
        print('잘못된 Parameter 요청. min_prop는 반드시 0 또는 0과 1사이의 값이어야합니다.')
        return 'None'

    if (type(min_ir_user_num) == int and min_ir_user_num < 0) or (type(min_ir_user_num) != int):
        print('잘못된 Parameter 요청. min_ir_user_num은 반드시 양의 정수 값이어야합니다.')
        return 'None'

    find_idx = sp.sparse.find(CS_mat == user_id)[1]
    user_idx = find_idx.item() if len(find_idx) > 0 else -1
    if user_idx == -1:
        print('추천 실패: 존재하지 않는 ID')
        return 'None'

    this_CF_mat = CF_mat[user_idx]
    wat_ani_list = []
    for ani_id, ani_idx in all_ani:
        if this_CF_mat[0,ani_idx] > 0:
            wat_ani_list.append(ani_id)

    wat_genre = {}
    for col in genre_cols:
        if ani_meta_df.loc[wat_ani_list, col].sum() > 0:
            wat_genre[col.split('_')[1]] = ani_meta_df.loc[wat_ani_list, col].sum() / ani_meta_df.loc[wat_ani_list, genre_cols].to_numpy().sum()

    sorted_wat_genre = {genre: ratio for genre, ratio in sorted(wat_genre.items(), key=lambda item: item[1], reverse=True) if ratio > min_prop}
    if len(sorted_wat_genre) == 0:
        if verbose:
            print('추천 실패: 가장 높은 시청 비율 ({0})이 min_prop 파라미터 값인 ({1})보다 작습니다. min_prop 파라미터가 지나치게 높게 설정되어 있습니다.'.format(list(sorted(wat_genre.values(), reverse=True))[0], min_prop))
        return 'None'

    dominate_genre = list({k: sorted_wat_genre[k] for k in list(sorted_wat_genre)[:find_opt_key_num(sorted_wat_genre, cut=cut)]}.keys())
    if len(dominate_genre) == 0:
        print('추천 실패: 장르 선별 과정에서 모든 장르가 탈락되었습니다. cut 파라미터와 min_prop 파라미터 값을 적절히 조정해야 합니다.')
        return 'None'

    if verbose:
        print('총 {0}개의 애니메이션을 시청한 유저입니다. 시청한 애니메이션 목록은 다음과 같습니다.'.format(len(wat_ani_list)))
        print(", ".join(ani_meta_df.loc[ani_meta_df[ani_meta_df.index.isin(wat_ani_list)].index]['sris_nm'].tolist()))
        print('시청한 애니메이션은 총 {0}개의 서로 다른 장르로 분류됩니다. 장르별 비율은 다음과 같습니다.'.format(len(wat_genre)))
        print(', '.join(f'{get_kor_genre_name(key)}: {"{:.3f}".format(value)}' for key, value in wat_genre.items()))
        genre_message = '상위 누적 {0}%의 장르만 선택합니다. '.format(cut * 100) if (cut > 0 and cut < 1) else '자주 보는 장르를 고려하지 않고 모든 장르를 추천합니다. '
        genre_message = genre_message + ('최소 {0}% 미만으로 시청된 장르는 제외됩니다. '.format(min_prop * 100) if min_prop > 0 else '')
        genre_message = genre_message + ('이 기준에 따라 {0}개의 카테고리가 제외되었습니다.'.format(len(wat_genre) - len(dominate_genre)) if len(wat_genre) - len(dominate_genre) > 0  else ('이 기준에 따라 어느 장르의 카테고리도 제외되지 않았습니다.' if (cut > 0 and cut < 1) or (min_prop > 0) else ''))
        print(genre_message)
        print('최종적으로 고려하는 장르는 다음과 같습니다.')
        print(', '.join(f'{get_kor_genre_name(value)}' for value in dominate_genre))

    recommend_candidates = [(ani_id, ani_idx) for ani_id, ani_idx in all_ani 
                           if (ani_id not in wat_ani_list) and (ani_meta_df.loc[ani_id, 'gnr'] in dominate_genre)]
    if verbose:
        print('{0}개의 장르에서 이 유저가 시청하지 않은 애니메이션은 {1}개입니다.'.format(len(dominate_genre), len(recommend_candidates)))
 
    cf_info = get_cf_feasibility(user_idx, std=std)
    if cf_info['FAIL'] == 1:
        if verbose:
            print('추천 실패: Infeasible on CF (' + cf_info['FAIL_REASON'] + ')')
        return 'None'
    cf_cs = list(cf_info['OVERLAP_DATA'].keys())
    cf_users_idx = np.where(np.isin(CS_mat, cf_cs))[0]

    if verbose:
        print('CF에 이용할 다른 유저들의 정보를 불러왔습니다. 요약 통계량은 아래와 같습니다.')
        print('다른 유저 수: {0}명, 겹친 애니메이션 최소 개수: {1}개, 겹친 애니메이션의 최대 개수: {2}개'.format(cf_info['SUMMARY_STAT']['NUM_CORR_USER'], cf_info['SUMMARY_STAT']['MIN_OVERLAP'], cf_info['SUMMARY_STAT']['MAX_OVERLAP']))

    fail = []
    wat_ani_users = {}
    for ani_id, ani_idx in recommend_candidates:
        wat_users_idx = np.intersect1d(cf_users_idx, positive_rating_idx_dict[ani_idx])
        wat_users = [user for user in CS_mat[wat_users_idx].tolist() if user != user_id]

        if len(wat_users) > 0:
            fail.append(0)
            wat_users_rating = CF_mat[wat_users_idx, ani_idx].toarray().reshape(-1).tolist()
            wat_ani_users[ani_id] = dict(zip(wat_users, wat_users_rating))
        else:
            fail.append(1)

    if sum(fail) == len(recommend_candidates):
        if verbose:
            print('추천 실패: Correlation 계산에 포함된 다른 유저들 중, 추천 후보 애니메이션을 시청한 유저가 존재하지 않음.')
        return 'None'

    if verbose:
        print('추천 후보 애니메이션 중에서, CF에 이용할 다른 유저들이 시청한 애니메이션 개수는 총 {0}개입니다.'.format(len(fail) - sum(fail)))

    fail = []
    ratings = {}
    for ani_id, wat_rating_info in wat_ani_users.items():
        user_id_list = list(wat_rating_info.keys())
        if len(user_id_list) < min_ir_user_num:
            fail.append(1)
            continue
        else:
            fail.append(0)
        corrcoefs = np.array([cf_info['OVERLAP_DATA'][id] for id in user_id_list])
        if std:
            explict_ratings = np.array([(wat_rating_info[id] - cf_info['RATING_STAT']['OTHER_RATING'][id][0]) / cf_info['RATING_STAT']['OTHER_RATING'][id][1] for id in user_id_list])
            if np.sum(np.abs(corrcoefs)) == 0: # 모든 Weight들이 0
                this_ir = 0
            else:
                this_ir = np.sum(corrcoefs * explict_ratings) / np.sum(np.abs(corrcoefs))
            this_ir = this_ir * cf_info['RATING_STAT']['THIS_RATING'][1] + cf_info['RATING_STAT']['THIS_RATING'][0]
        else:
            explict_ratings = np.array([wat_rating_info[id] for id in user_id_list])
            this_ir = np.sum(corrcoefs * explict_ratings) / np.sum(np.abs(corrcoefs))
        ratings[ani_id] = this_ir

    if sum(fail) == len(wat_ani_users):
        if verbose:
            print('추천 실패: 추천 후보 애니메이션을 시청한 유저가 충분히 존재하지 않음.')
        return 'None'

    sorted_ratings = {ani_id: rating for ani_id, rating in sorted(ratings.items(), key=lambda item: item[1], reverse=True)}

    if verbose:
        print('표준화를 하여 계산되었습니다.' if std == True else '표준화를 하지 않고 계산되었습니다.')
        print('최종적으로 추천 후보 애니메이션 개수는 총 {0}개입니다.'.format(len(fail) - sum(fail)) + (' 즉, 추천 후보 애니메이션을 시청한 유저가 충분하지 않은 애니메이션의 개수는 {0}개입니다.'.format(sum(fail)) if sum(fail) > 0 else '')) 
        print('최종 추천 후보 애니메이션들의 기대 평점은 아래와 같습니다.')
        for (ani_id, rating) in sorted_ratings.items():
            print(ani_id + ': ' + str(rating))

    if output_len > len(sorted_ratings):
        if verbose:
            print('출력하고자 하는 애니메이션의 개수가 최종 후보 애니메이션의 개수보다 크기 때문에, 총 {0}개의 추천 애니메이션만 출력됩니다.'.format(len(sorted_ratings)))
        output_len = len(sorted_ratings)

    output_dict = {k: sorted_ratings[k] for k in list(sorted_ratings)[:output_len]}
    if verbose:
        print('출력되는 추천 애니메이션의 이름과 그 기대 평점은 아래와 같습니다.')
        for (ani_id, rating) in output_dict.items():
            print(ani_meta_df.loc[ani_meta_df.index == ani_id]['sris_nm'].values[0] + ': ' + str(rating))
    if output_len == 1:
        return list(output_dict.keys())[0]
    return ", ".join(list(output_dict.keys()))

In [ ]:
draw_random_user(cf_feasibility_df, n=5, conds='NUM_CORR_USER >= 100, AVG_OVERLAP > 3')

In [ ]:
recommendation(7118170337, min_prop=0.1, cut=1, std=True, output_len=2, verbose=True)

## CF 추천 가능한 유저 중, 실제로 추천이 가능한 유저 비율 추정

### CF Feasbility Data 로드

In [ ]:
cf_feasibility_df = pd.read_csv('./cf_feasibility.csv')

### Feasible하지 않은 유저 데이터 삭제 후 남은 유저 ID 리스트 저장

In [ ]:
cf_feasibility_df = cf_feasibility_df[cf_feasibility_df['FAIL'] == 0].reset_index(drop=True)
id_list = cf_feasibility_df['svc_mgmt_num'].tolist()

### 20분위 수 칼럼 생성 

In [ ]:
cf_feasibility_df['QUANTILE'] = pd.qcut(cf_feasibility_df['NUM_CORR_USER'], 20, labels=False)

### 분위 수별 통계량

In [ ]:
quantile_stat = {}
for i in range(20):
    q_df = cf_feasibility_df[cf_feasibility_df['QUANTILE'] == i]
    quantile_stat[i] = {'OBS': len(q_df), 'PROP': len(q_df) / len(cf_feasibility_df), 'MIN': q_df['NUM_CORR_USER'].min(), 'MAX': q_df['NUM_CORR_USER'].max(), 'MEAN': q_df['NUM_CORR_USER'].mean()} 

In [ ]:
for i in range(20):
    print('{0}분위 -  관측치: {1}, 비율: {2}, 최솟값: {3}, 최댓값: {4}, 평균: {5}'.format(i + 1, quantile_stat[i]['OBS'], quantile_stat[i]['PROP'], quantile_stat[i]['MIN'], quantile_stat[i]['MAX'], quantile_stat[i]['MEAN']))

### 분위수별 추천 가능한 유저 수 비율 추론 함수
### Input
<code>cf_fs_df</code>: CF Feasible한 유저들에 대한 Feasiblity 통계량을 모아둔 데이터입니다.  
<code>sample_size</code>: 임의 표본을 구성할 표본의 크기입니다. 비율 값이며, 기본값은 0.1(10%)입니다.  
<code>seed</code>: Numpy의 random.randint에 입력될 시드값입니다. 기본값은 1230234입니다.  
<code>out</code>: CF Feasible한 유저들에 임의 추출 여부와 추천 가능 여부 데이터를 추가한 정보를 추출할지 결정합니다. 기본값은 True입니다.
<code>verbose</code>: 함수 실행의 각 과정을 상세히 출력합니다. 기본값은 False입니다.

In [ ]:
def infer_prop(cf_fs_df, sample_size=0.1, seed=1230234, out=True, verbose=False, tot_res=None):
    if verbose:
        print('설정된 표본 크기: {0}%, 시드값: {1}'.format(sample_size * 100, seed) + ', CSV 추출 여부: ' + '추출' if out else '추출 안함')

    if tot_res == None:
        tot_res = {}
        if verbose:
            print('표본 크기 {0}%의 임의 표본을 분위별로 생성 후, 각 표본마다 추천 가능성을 기록합니다.'.format(sample_size * 100))
        for i in range(20):
            if verbose:
                print('현재 {0}분위 유저들에 대해 추천 가능 여부를 기록하고 있습니다.'.format(i + 1))
            tot_res[i] = {}
            q_df_sample = cf_fs_df[cf_fs_df['QUANTILE'] == i].sample(frac=sample_size, random_state=np.random.randint(seed))
            user_id_list = q_df_sample['svc_mgmt_num'].tolist()
            for user_id in tqdm(user_id_list):
                res = 1 if recommendation(user_id) != 'None' else 0
                tot_res[i][user_id] = res

    if verbose:
        print('각 분위별 포함된 유저 수와 추천 가능한 유저 수는 다음과 같습니다:')
        for i in range(20):
            recommendation_num = sum([val for val in tot_res[i].values()])
            total_num = len(tot_res[i])
            prop = recommendation_num / total_num
            print('{0}분위 - 추천 가능 유저 수: {1}, 포함된 유저 수: {2}, 비율: {3}'.format(i + 1, recommendation_num, total_num, prop))

        print('Feasibility 데이터에 임의 표본 포함 여부 및 추천 가능 여부를 기록합니다.')

    selected_user_list = []
    selected_cf_recommendation_list = []
    for i in range(20):
        selected_user_list = selected_user_list + ([id for id in tot_res[i].keys()])
        selected_cf_recommendation_list = selected_cf_recommendation_list + ([SoF for SoF in tot_res[i].values()])

    if out:
        new_cf_fs_df = cf_feasibility_df.copy(True)
        new_cf_fs_df = new_cf_fs_df.drop(columns=['FAIL', 'FAIL_REASON'])
        new_cf_fs_df = new_cf_fs_df.set_index('svc_mgmt_num')
        new_cf_fs_df['SAMPLE_SELECTED'] = 0
        new_cf_fs_df['CF_RECOMMANDATION'] = np.nan
        new_cf_fs_df.loc[selected_user_list, 'SAMPLE_SELECTED'] = 1
        new_cf_fs_df.loc[selected_user_list, 'CF_RECOMMANDATION'] = selected_cf_recommendation_list
        new_cf_fs_df.to_csv('./cf_recommendation_user.csv')

    prop = {}
    for i in range(20):
        prop[i] = sum(tot_res[i].values()) / len(tot_res[i])

    if verbose:
        include_non_one_elem = [key + 1 for key in prop.keys() if prop[key] != 1]
        if len(include_non_one_elem) == 0:
            print('해당 임의 표본 내 모든 분위의 유저들이 추천이 가능합니다.')
        else:
            print('추천이 가능하지 않은 유저가 포함된 분위: ' + '분위, '.join(map(str, include_non_one_elem)) + '분위')
            print('해당 분위의 추천 가능한 유저의 비율 추정치 및 95% 신뢰구간은 다음과 같습니다:')
            for i in include_non_one_elem:
                print('{0}분위'.format(i))
                print('추천 가능한 유저 비율 추정치: {0}'.format(prop[i - 1]))
                left = prop[i - 1] - 1.96 * np.sqrt(prop[i - 1] * (1 - prop[i - 1]) / len(tot_res[i - 1]))
                right = prop[i - 1] + 1.96 * np.sqrt(prop[i - 1] * (1 - prop[i - 1]) / len(tot_res[i - 1]))
                if left < 0:
                    left = 0
                if right > 1:
                    right = 1
                print('95% 신뢰구간: [{0}, {1}]'.format(left, right))
            one = 0
            for i in range(20):
                one = one + sum([val for val in tot_res[i].values()])
            print('추천 가능 유저의 전체 비율: {0}'.format(one / sum([len(tot_res[i]) for i in range(20)])))

In [ ]:
infer_prop(cf_feasibility_df, verbose=True) # 표본 크기 0.1 기준 실행시간 약 1 ~ 2시간 소요

\[1]: https://stackoverflow.com/questions/13077527/is-there-a-numpy-delete-equivalent-for-sparse-matrices  
\[2]: https://gist.github.com/sumartoyo/edba2eee645457a98fdf046e1b4297e4  
\[3]: https://stackoverflow.com/questions/33650188/efficient-pairwise-correlation-for-two-matrices-of-features